# LeetCode #6 — Zigzag Conversion: mô phỏng đường đi và gom theo hàng

**Mức độ:** Medium · **Chủ đề:** String, Simulation

Skeleton trên LeetCode:

```python
class Solution:
    def convert(self, s: str, numRows: int) -> str:
```

Bài này không cần cấu trúc dữ liệu phức tạp. Cái khó là **đọc hiểu hình vẽ** và **biến hình vẽ thành quy luật**.

Mục tiêu của notebook là luyện kỹ năng **mô phỏng (simulation)**: theo dõi một "con trỏ hàng" đi lên đi xuống, và **gom ký tự vào đúng nhóm** thay vì vẽ cả bảng.

---

## Sau notebook này, bạn sẽ nắm được

1. Cách đọc hình zigzag và gán mỗi ký tự vào một **hàng**.
2. Cách tạo danh sách các hàng, và cái bẫy `[[]] * n` trong Python.
3. Cách giải trực giác bằng **lưới 2 chiều**, và vì sao nó lãng phí.
4. Cách giải chính: biến `row` nảy lên nảy xuống với `step = +1 / -1`.
5. Cách giải bằng **công thức chu kỳ** `cycle = 2 * numRows - 2`.
6. Trường hợp đặc biệt `numRows = 1`.
7. Bài tập vận dụng, gồm cả bài toán **ngược** (giải mã zigzag).

# 1. Bài toán là gì?

Viết chuỗi `s` theo hình **zigzag** trên `numRows` hàng: đi **thẳng xuống**, rồi đi **chéo lên**, rồi lại thẳng xuống, ...

Sau đó **đọc từng hàng** từ trên xuống, từ trái sang phải, ghép lại thành chuỗi kết quả.

**Constraints (theo đề):**

- `1 <= s.length <= 1000`
- `s` gồm chữ cái tiếng Anh (hoa và thường), `','` và `'.'`
- `1 <= numRows <= 1000`

---

## Ví dụ 1: `s = "PAYPALISHIRING"`, `numRows = 3`

```text
P   A   H   N
A P L S I I G
Y   I   R
```

Đọc từng hàng: `"PAHN" + "APLSIIG" + "YIR"` = **`"PAHNAPLSIIGYIR"`**

## Ví dụ 2: cùng chuỗi, `numRows = 4`

```text
P     I     N
A   L S   I G
Y A   H R
P     I
```

Kết quả: **`"PINALSIGYAHRPI"`**

## Ví dụ 3

```text
s = "A", numRows = 1  ->  "A"
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Chuỗi (string) trong Python
2. Ma trận (mảng hai chiều) trong Python

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Chuỗi (string) trong Python

### Chuỗi là bất biến

`str` **không sửa được** tại chỗ: `s[0] = "x"` gây lỗi. Mọi "sửa" đều tạo chuỗi mới. Vì vậy:

- Ghép nhiều mảnh: gom vào **list** rồi `"".join(list)` (`O(n)`), tránh `s += ...` trong vòng lặp (có thể `O(n²)`).
- Cần sửa từng ký tự: `chars = list(s)`, sửa, rồi `"".join(chars)`.

In [1]:
s_str = "hello"
try:
    s_str[0] = "H"
except TypeError as e:
    print("Sửa trực tiếp:", e)
chars_str = list(s_str); chars_str[0] = "H"
print("qua list:", "".join(chars_str))
manh = []
for i in range(5):
    manh.append(str(i))
print("join:", "-".join(manh))

Sửa trực tiếp: 'str' object does not support item assignment
qua list: Hello
join: 0-1-2-3-4


### Cắt chuỗi (slicing) và các thao tác hay dùng

| Biểu thức | Ý nghĩa |
|---|---|
| `s[i:j]` | ký tự từ `i` tới `j−1` (tạo chuỗi mới, `O(j−i)`) |
| `s[::-1]` | chuỗi đảo ngược |
| `s.split()` | tách theo khoảng trắng (bỏ khoảng trắng thừa) |
| `s.split(",")` | tách theo dấu phẩy (giữ phần rỗng) |
| `s.strip()` | bỏ khoảng trắng hai đầu |
| `s.startswith(p)`, `s.find(p)` | kiểm tra tiền tố, tìm vị trí |
| `c.isdigit()`, `c.isalpha()`, `c.isalnum()`, `c.isspace()` | phân loại ký tự |
| `c.lower()`, `c.upper()` | đổi hoa/thường |

In [2]:
t_str = "  Xin chào   thế giới  "
print(repr(t_str.split()), "|", repr(t_str.strip()), "|", repr("a,,b".split(",")))
print("đảo:", "abcdef"[::-1], "| cắt [1:4]:", "abcdef"[1:4], "| 'abc'.find('c') =", "abc".find("c"))
print([(c, c.isdigit(), c.isalpha()) for c in "a1 "])

['Xin', 'chào', 'thế', 'giới'] | 'Xin chào   thế giới' | ['a', '', 'b']
đảo: fedcba | cắt [1:4]: bcd | 'abc'.find('c') = 2
[('a', False, True), ('1', True, False), (' ', False, False)]


### Ký tự và mã số: `ord` / `chr`

Mỗi ký tự có một mã số (Unicode). `ord("a") = 97`, `chr(97) = "a"`. Mẹo: chữ số `c` → giá trị `ord(c) - ord("0")`; chữ thường `c` → chỉ số 0..25 là `ord(c) - ord("a")`.

In [3]:
print("ord('a') =", ord("a"), "| chr(98) =", chr(98))
print("'7' → 7:", ord("7") - ord("0"), "| 'd' → chỉ số:", ord("d") - ord("a"))
dem26 = [0] * 26
for c in "banana":
    dem26[ord(c) - ord("a")] += 1
print("đếm 26 chữ cái của 'banana' (a, b, n):", dem26[0], dem26[1], dem26[13])

ord('a') = 97 | chr(98) = b
'7' → 7: 7 | 'd' → chỉ số: 3
đếm 26 chữ cái của 'banana' (a, b, n): 3 1 2


### Gom nhóm ký tự liên tiếp

`itertools.groupby` gom các phần tử **liên tiếp bằng nhau** — tiện cho mã hóa độ dài loạt (run-length): `"aaabcc" → 3a 1b 2c`. Tự viết bằng hai con trỏ: `i` ở đầu nhóm, `j` chạy tới khi ký tự khác.

In [4]:
from itertools import groupby
print([(k, len(list(g))) for k, g in groupby("aaabcc")])

def run_length(s):
    out, i = [], 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        out.append(f"{j - i}{s[i]}")
        i = j
    return "".join(out)
print("run_length('aaabcc') =", run_length("aaabcc"))

[('a', 3), ('b', 1), ('c', 2)]
run_length('aaabcc') = 3a1b2c


Lỗi hay gặp: `split(" ")` sinh phần rỗng khi có nhiều dấu cách; so sánh chuỗi số `"10" < "9"` là **True** (so từ điển, không phải số); nối chuỗi trong vòng lặp lớn; quên chuỗi rỗng.

## 2.2 Ma trận (mảng hai chiều) trong Python

### Tạo và truy cập

Ma trận `m × n` là list gồm `m` hàng, mỗi hàng là list `n` phần tử. Ô hàng `i`, cột `j` là `M[i][j]`; số hàng `len(M)`, số cột `len(M[0])`.

```text
        j=0 j=1 j=2
i=0   [  1,  2,  3 ]
i=1   [  4,  5,  6 ]
```

In [5]:
M_mx = [[1, 2, 3],
        [4, 5, 6]]
print("M[1][2] =", M_mx[1][2], "| số hàng:", len(M_mx), "| số cột:", len(M_mx[0]))

dung_mx = [[0] * 3 for _ in range(2)]    # ĐÚNG: mỗi hàng là list riêng
sai_mx = [[0] * 3] * 2                   # SAI: 2 hàng là cùng một list
dung_mx[0][0] = sai_mx[0][0] = 7
print("tạo bằng for:", dung_mx, "| nhân list:", sai_mx)

M[1][2] = 6 | số hàng: 2 | số cột: 3
tạo bằng for: [[7, 0, 0], [0, 0, 0]] | nhân list: [[7, 0, 0], [7, 0, 0]]


### Duyệt và các phép biến đổi cơ bản

- Duyệt theo hàng: `for i in range(m): for j in range(n)`.
- **Chuyển vị** (transpose): `T[j][i] = M[i][j]` — hàng thành cột. Trong Python: `list(zip(*M))`.
- **Lật ngang** một hàng: `row[::-1]`; lật dọc: đảo thứ tự hàng `M[::-1]`.
- Xoay 90° theo chiều kim đồng hồ = chuyển vị rồi lật từng hàng.

In [6]:
def in_mt(A):
    for row in A:
        print("   ", " ".join(f"{v:>2}" for v in row))

A_mx = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print("gốc:"); in_mt(A_mx)
T_mx = [list(r) for r in zip(*A_mx)]
print("chuyển vị:"); in_mt(T_mx)
print("xoay 90° phải (chuyển vị + lật hàng):"); in_mt([r[::-1] for r in T_mx])

gốc:
     1  2  3
     4  5  6
     7  8  9
chuyển vị:
     1  4  7
     2  5  8
     3  6  9
xoay 90° phải (chuyển vị + lật hàng):
     7  4  1
     8  5  2
     9  6  3


### Ô kề và mảng hướng

Bốn ô kề của `(r, c)`: lên `(r−1, c)`, xuống `(r+1, c)`, trái `(r, c−1)`, phải `(r, c+1)`. Viết thành **mảng hướng** để khỏi lặp code, và luôn **kiểm tra biên** `0 <= nr < m and 0 <= nc < n`.

In [7]:
HUONG = [(-1, 0), (1, 0), (0, -1), (0, 1)]       # lên, xuống, trái, phải
def o_ke(r, c, m, n):
    return [(r + dr, c + dc) for dr, dc in HUONG if 0 <= r + dr < m and 0 <= c + dc < n]
print("ô kề của (0, 0) trong lưới 3×3:", o_ke(0, 0, 3, 3))
print("ô kề của (1, 1):", o_ke(1, 1, 3, 3))

ô kề của (0, 0) trong lưới 3×3: [(1, 0), (0, 1)]
ô kề của (1, 1): [(0, 1), (2, 1), (1, 0), (1, 2)]


### Đánh số ô thành một chỉ số và ngược lại

Ma trận `m × n` "trải phẳng" theo hàng: ô `(i, j)` ↔ chỉ số `k = i * n + j`; ngược lại `i, j = divmod(k, n)`. Hữu ích để tìm nhị phân trên ma trận hoặc lưu ô vào set.

In [8]:
n_cols = 4
for k in [0, 5, 11]:
    print(f"k = {k:>2} → (i, j) = {divmod(k, n_cols)} → k = {divmod(k, n_cols)[0] * n_cols + divmod(k, n_cols)[1]}")

k =  0 → (i, j) = (0, 0) → k = 0
k =  5 → (i, j) = (1, 1) → k = 5
k = 11 → (i, j) = (2, 3) → k = 11


Lỗi hay gặp: `[[0]*n]*m`; nhầm `m` (hàng) với `n` (cột) khi ma trận không vuông; quên kiểm tra biên; sửa ma trận trong lúc vẫn cần giá trị cũ để tính ô khác.

## 2.3 Áp dụng vào bài này

Chuỗi được "viết" theo hình zigzag lên `numRows` **hàng**, rồi đọc lần lượt từng hàng. Ta mô phỏng bằng một **list các hàng** (giống một ma trận có hàng dài ngắn khác nhau), mỗi ký tự được nối vào đúng hàng của nó, đổi chiều khi chạm hàng đầu/cuối. Kết quả ghép bằng `"".join(...)` thay vì cộng chuỗi trong vòng lặp.

# 3. Gán mỗi ký tự vào một hàng

Bỏ qua khoảng trắng trong hình vẽ. Điều duy nhất quan trọng là: **mỗi ký tự nằm ở hàng nào?**

Với `numRows = 3`, ghi số hàng của từng ký tự theo thứ tự trong `s`:

| Chỉ số | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 | 13 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Ký tự | P | A | Y | P | A | L | I | S | H | I | R | I | N | G |
| Hàng | 0 | 1 | 2 | 1 | 0 | 1 | 2 | 1 | 0 | 1 | 2 | 1 | 0 | 1 |

Dãy hàng: `0, 1, 2, 1, 0, 1, 2, 1, 0, ...` — đi xuống tới đáy, **nảy** lên tới đỉnh, rồi lại nảy xuống.

Gom ký tự theo hàng:

- Hàng 0: `P A H N`
- Hàng 1: `A P L S I I G`
- Hàng 2: `Y I R`

Ghép lại chính là đáp án. **Ta không cần vẽ hình!**

In [9]:
s = "PAYPALISHIRING"
rows_by_hand = [0, 1, 2, 1, 0, 1, 2, 1, 0, 1, 2, 1, 0, 1]

for r in range(3):
    chars = [c for c, row in zip(s, rows_by_hand) if row == r]
    print(f"Hàng {r}: {''.join(chars)}")

Hàng 0: PAHN
Hàng 1: APLSIIG
Hàng 2: YIR


`zip(a, b)` ghép từng cặp phần tử cùng vị trí của hai dãy:

In [10]:
print(list(zip("PAY", [0, 1, 2])))

[('P', 0), ('A', 1), ('Y', 2)]


# 4. Nền tảng Python: danh sách các hàng

Ta cần `numRows` "cái giỏ", mỗi giỏ chứa ký tự của một hàng.

Cách đơn giản: một list gồm `numRows` chuỗi rỗng. Dùng `+=` để nối ký tự vào cuối chuỗi của một hàng.

In [11]:
rows = [""] * 3
rows[0] += "P"
rows[1] += "A"
rows[0] += "A"
print(rows)
print("".join(rows))   # ghép các hàng lại

['PA', 'A', '']
PAA


## Cái bẫy: `[[]] * n`

Nếu muốn mỗi hàng là một **list** (thay vì chuỗi), đừng viết `[[]] * 3`. Cách đó tạo ra **3 tham chiếu tới cùng một list** — sửa một hàng là sửa cả ba!

Chuỗi (`""`) thì an toàn, vì `rows[0] += "P"` tạo **chuỗi mới** và gán lại vào `rows[0]`.

In [12]:
bad = [[]] * 3
bad[0].append("P")
print("Sai:", bad)          # cả 3 hàng đều có 'P'

good = [[] for _ in range(3)]
good[0].append("P")
print("Đúng:", good)

Sai: [['P'], ['P'], ['P']]
Đúng: [['P'], [], []]


# 5. Cách trực giác: vẽ thật lưới 2 chiều

Cách "đúng nghĩa đen" nhất: tạo một lưới `numRows × số_cột`, đặt từng ký tự vào ô của nó, rồi đọc theo hàng (bỏ ô trống).

Quy tắc di chuyển:

- Đang **đi xuống**: `row += 1`, giữ nguyên cột.
- Đang **đi chéo lên**: `row -= 1`, `col += 1`.

Số cột không vượt quá `len(s)`, nên ta tạo lưới rộng `len(s)` cho đơn giản.

In [13]:
def build_grid(s: str, numRows: int):
    grid = [[" "] * len(s) for _ in range(numRows)]
    row, col = 0, 0
    going_down = True

    for c in s:
        grid[row][col] = c
        if numRows == 1:
            col += 1
        elif going_down:
            if row == numRows - 1:      # chạm đáy -> chuyển sang đi chéo lên
                going_down = False
                row -= 1
                col += 1
            else:
                row += 1
        else:
            if row == 0:                # chạm đỉnh -> chuyển sang đi xuống
                going_down = True
                row += 1
            else:
                row -= 1
                col += 1
    return grid


def print_grid(grid):
    for line in grid:
        print("".join(line).rstrip())


print_grid(build_grid("PAYPALISHIRING", 3))
print()
print_grid(build_grid("PAYPALISHIRING", 4))

P A H N
APLSIIG
Y I R

P  I  N
A LS IG
YA HR
P  I


Hình in ra "gọn" hơn hình trong đề (mỗi cột chỉ rộng một ký tự, không có khoảng cách thêm), nhưng **vị trí tương đối** của các ký tự giống hệt. Đọc từng hàng, bỏ ô trống:

In [14]:
def convert_grid(s: str, numRows: int) -> str:
    grid = build_grid(s, numRows)
    return "".join(c for line in grid for c in line if c != " ")

print(convert_grid("PAYPALISHIRING", 3))
print(convert_grid("PAYPALISHIRING", 4))
print(convert_grid("A", 1))

PAHNAPLSIIGYIR
PINALSIGYAHRPI
A


## Vấn đề của cách này

- Lưới có `numRows × len(s)` ô, nhưng **chỉ `len(s)` ô có ký tự**. Với `numRows = 1000`, `len(s) = 1000`, ta tạo **một triệu ô** để dùng một nghìn.
- Cột (`col`) hoàn toàn **không ảnh hưởng tới đáp án**: trong cùng một hàng, ký tự nào được viết trước thì nằm bên trái.
- Code xử lý cột khá rối, dễ sai.

> Bỏ hẳn cột. Chỉ theo dõi **hàng**, và nối ký tự vào giỏ của hàng đó.

# 6. Lời giải chính: con trỏ hàng nảy lên nảy xuống

Dùng hai biến:

- `row`: hàng hiện tại.
- `step`: hướng đi, `+1` là đi xuống, `-1` là đi lên.

Với mỗi ký tự:

1. Nối ký tự vào `rows[row]`.
2. Nếu đang ở **đỉnh** (`row == 0`) thì đổi hướng thành đi xuống; nếu ở **đáy** (`row == numRows - 1`) thì đổi thành đi lên.
3. `row += step`.

Thử nhanh dãy hàng mà cách này tạo ra:

In [15]:
def row_sequence(n: int, numRows: int):
    seq = []
    row, step = 0, 1
    for _ in range(n):
        seq.append(row)
        if row == 0:
            step = 1
        elif row == numRows - 1:
            step = -1
        row += step
    return seq

print(row_sequence(14, 3))
print(row_sequence(14, 4))

[0, 1, 2, 1, 0, 1, 2, 1, 0, 1, 2, 1, 0, 1]
[0, 1, 2, 3, 2, 1, 0, 1, 2, 3, 2, 1, 0, 1]


Dãy cho `numRows = 3` khớp đúng bảng tính tay ở mục 3.

## Trường hợp đặc biệt: `numRows = 1`

Khi chỉ có một hàng, `row = 0` vừa là đỉnh vừa là đáy. Code trên sẽ đặt `step = 1` (vì kiểm tra `row == 0` trước) rồi `row` thành `1` — **vượt quá** số hàng!

Cách xử lý gọn nhất: nếu `numRows == 1` (hoặc `numRows >= len(s)`), chuỗi zigzag chính là chuỗi gốc, trả về `s` ngay.

In [16]:
print(row_sequence(4, 1))   # [0, 1, 2, 3] — sai: chỉ có hàng 0!

[0, 1, 2, 3]


In [17]:
def convert_simulation(s: str, numRows: int) -> str:
    if numRows == 1 or numRows >= len(s):
        return s

    rows = [""] * numRows
    row, step = 0, 1

    for c in s:
        rows[row] += c
        if row == 0:
            step = 1
        elif row == numRows - 1:
            step = -1
        row += step

    return "".join(rows)

print(convert_simulation("PAYPALISHIRING", 3))
print(convert_simulation("PAYPALISHIRING", 4))
print(convert_simulation("A", 1))

PAHNAPLSIIGYIR
PINALSIGYAHRPI
A


Vì sao `numRows >= len(s)` cũng trả về `s`? Khi đó ta chỉ đi **thẳng xuống**, mỗi ký tự một hàng riêng, chưa kịp nảy lên. Đọc từng hàng từ trên xuống lại ra đúng `s`. (Điều kiện này không bắt buộc cho tính đúng — chỉ là lối tắt.)

# 7. Mô phỏng từng bước

`s = "PAYPALISHIRING"`, `numRows = 3`. Theo dõi `row`, `step` và nội dung các hàng sau mỗi ký tự:

In [18]:
def trace_convert(s: str, numRows: int) -> str:
    rows = [""] * numRows
    row, step = 0, 1
    for i, c in enumerate(s):
        rows[row] += c
        note = ""
        if row == 0:
            step = 1
            note = "đỉnh -> đi xuống"
        elif row == numRows - 1:
            step = -1
            note = "đáy  -> đi lên"
        print(f"i={i:2d} c={c} vào hàng {row}  rows={rows}  {note}")
        row += step
    return "".join(rows)

print("Kết quả:", trace_convert("PAYPALISHIRING", 3))

i= 0 c=P vào hàng 0  rows=['P', '', '']  đỉnh -> đi xuống
i= 1 c=A vào hàng 1  rows=['P', 'A', '']  
i= 2 c=Y vào hàng 2  rows=['P', 'A', 'Y']  đáy  -> đi lên
i= 3 c=P vào hàng 1  rows=['P', 'AP', 'Y']  
i= 4 c=A vào hàng 0  rows=['PA', 'AP', 'Y']  đỉnh -> đi xuống
i= 5 c=L vào hàng 1  rows=['PA', 'APL', 'Y']  
i= 6 c=I vào hàng 2  rows=['PA', 'APL', 'YI']  đáy  -> đi lên
i= 7 c=S vào hàng 1  rows=['PA', 'APLS', 'YI']  
i= 8 c=H vào hàng 0  rows=['PAH', 'APLS', 'YI']  đỉnh -> đi xuống
i= 9 c=I vào hàng 1  rows=['PAH', 'APLSI', 'YI']  
i=10 c=R vào hàng 2  rows=['PAH', 'APLSI', 'YIR']  đáy  -> đi lên
i=11 c=I vào hàng 1  rows=['PAH', 'APLSII', 'YIR']  
i=12 c=N vào hàng 0  rows=['PAHN', 'APLSII', 'YIR']  đỉnh -> đi xuống
i=13 c=G vào hàng 1  rows=['PAHN', 'APLSIIG', 'YIR']  
Kết quả: PAHNAPLSIIGYIR


Điểm cần chú ý trong trace:

- Ở `i = 2` (chữ `Y`), `row = 2` là đáy → `step` đổi thành `-1`.
- Ở `i = 4` (chữ `A`), `row = 0` là đỉnh → `step` đổi lại `+1`.
- Hàng giữa (hàng 1) được ghé thăm **hai lần mỗi chu kỳ**: một lần khi đi xuống, một lần khi đi lên.

# 8. Cách khác: công thức chu kỳ

Nhìn lại dãy hàng `0, 1, 2, 1 | 0, 1, 2, 1 | 0, ...`: nó **lặp lại** sau mỗi

```text
cycle = 2 * numRows - 2
```

ký tự (đi xuống `numRows` hàng, đi lên `numRows - 2` hàng giữa).

Với `numRows = 4`, `cycle = 6`. Chỉ số các ký tự trong mỗi hàng:

| Hàng | Chỉ số |
|---|---|
| 0 | 0, 6, 12 |
| 1 | 1, 5, 7, 11, 13 |
| 2 | 2, 4, 8, 10 |
| 3 | 3, 9 |

Quy luật cho hàng `r`, với mỗi chu kỳ bắt đầu tại `k` (`k = 0, cycle, 2*cycle, ...`):

- Luôn có ký tự tại `k + r` (lúc đi xuống).
- Nếu `r` là **hàng giữa** (không phải đỉnh hay đáy), còn có ký tự tại `k + cycle - r` (lúc đi lên).

Ví dụ hàng 1, chu kỳ `k = 0`: `0 + 1 = 1` và `0 + 6 - 1 = 5`. Khớp bảng!

In [19]:
numRows = 4
cycle = 2 * numRows - 2
n = len("PAYPALISHIRING")

for r in range(numRows):
    idx = []
    for k in range(0, n, cycle):
        if k + r < n:
            idx.append(k + r)
        if 0 < r < numRows - 1 and k + cycle - r < n:
            idx.append(k + cycle - r)
    print(f"Hàng {r}: {idx}")

Hàng 0: [0, 6, 12]
Hàng 1: [1, 5, 7, 11, 13]
Hàng 2: [2, 4, 8, 10]
Hàng 3: [3, 9]


In [20]:
def convert_cycle(s: str, numRows: int) -> str:
    if numRows == 1 or numRows >= len(s):
        return s

    n = len(s)
    cycle = 2 * numRows - 2
    out = []

    for r in range(numRows):
        for k in range(0, n, cycle):
            if k + r < n:
                out.append(s[k + r])
            if 0 < r < numRows - 1 and k + cycle - r < n:
                out.append(s[k + cycle - r])

    return "".join(out)

print(convert_cycle("PAYPALISHIRING", 3))
print(convert_cycle("PAYPALISHIRING", 4))

PAHNAPLSIIGYIR
PINALSIGYAHRPI


# 9. Phiên bản đúng format LeetCode

Ta nộp bản mô phỏng vì dễ đọc và khó sai hơn.

In [21]:
class Solution:
    def convert(self, s: str, numRows: int) -> str:
        if numRows == 1 or numRows >= len(s):
            return s

        rows = [""] * numRows
        row, step = 0, 1

        for c in s:
            rows[row] += c
            if row == 0:
                step = 1
            elif row == numRows - 1:
                step = -1
            row += step

        return "".join(rows)

In [22]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<28} PASS  {actual!r}")


sol = Solution()
check("Ví dụ 1: 3 hàng", sol.convert("PAYPALISHIRING", 3), "PAHNAPLSIIGYIR")
check("Ví dụ 2: 4 hàng", sol.convert("PAYPALISHIRING", 4), "PINALSIGYAHRPI")
check("Ví dụ 3: 1 hàng", sol.convert("A", 1), "A")
check("1 hàng, chuỗi dài", sol.convert("ABCD", 1), "ABCD")
check("2 hàng", sol.convert("ABCDEF", 2), "ACEBDF")
check("Số hàng = độ dài", sol.convert("ABC", 3), "ABC")
check("Số hàng > độ dài", sol.convert("AB", 1000), "AB")
check("Có dấu ',' và '.'", sol.convert("a,b.c", 2), "abc,.")

Ví dụ 1: 3 hàng              PASS  'PAHNAPLSIIGYIR'
Ví dụ 2: 4 hàng              PASS  'PINALSIGYAHRPI'
Ví dụ 3: 1 hàng              PASS  'A'
1 hàng, chuỗi dài            PASS  'ABCD'
2 hàng                       PASS  'ACEBDF'
Số hàng = độ dài             PASS  'ABC'
Số hàng > độ dài             PASS  'AB'
Có dấu ',' và '.'            PASS  'abc,.'


## Đối chiếu ba cách giải

In [23]:
import random

random.seed(6)
alphabet = "abcXYZ,."
for _ in range(2000):
    s = "".join(random.choice(alphabet) for _ in range(random.randint(1, 30)))
    k = random.randint(1, 12)
    expected = convert_grid(s, k)
    assert sol.convert(s, k) == expected, (s, k)
    assert convert_cycle(s, k) == expected, (s, k)

print("2000 test ngẫu nhiên: lưới 2D, mô phỏng và công thức chu kỳ đều khớp")

2000 test ngẫu nhiên: lưới 2D, mô phỏng và công thức chu kỳ đều khớp


# 10. Vì sao thuật toán đúng?

1. **Đúng hàng:** biến `row` sinh ra đúng dãy `0, 1, …, numRows-1, numRows-2, …, 1, 0, 1, …` — đây chính là đường đi zigzag (xuống thẳng, lên chéo).
2. **Đúng thứ tự trong một hàng:** trong hình zigzag, ký tự viết trước luôn nằm **bên trái** (hoặc cùng cột nhưng ở hàng khác). Ta duyệt `s` từ trái sang phải và **nối vào cuối** mỗi hàng, nên thứ tự trong hàng được giữ nguyên.
3. **Đúng thứ tự đọc:** `"".join(rows)` ghép hàng 0, hàng 1, … đúng như đề yêu cầu "đọc từng hàng từ trên xuống".

# 11. Độ phức tạp

Gọi `n = len(s)`.

| Cách làm | Thời gian | Bộ nhớ | Ghi chú |
|---|---|---|---|
| Lưới 2D | O(numRows · n) | O(numRows · n) | Lãng phí ô trống |
| Mô phỏng hàng | O(n) | O(n) | Mỗi ký tự xử lý một lần |
| Công thức chu kỳ | O(n) | O(n) | Mỗi ký tự được lấy đúng một lần |

Một chi tiết nhỏ: `rows[row] += c` tạo chuỗi mới mỗi lần. Python thường tối ưu được thao tác này, và với `n ≤ 1000` thì không đáng lo. Nếu muốn chắc chắn `O(n)`, dùng list các list rồi `"".join` ở cuối (xem Bài tập 4).

# 12. Những lỗi dễ mắc

## Lỗi 1: Quên trường hợp `numRows = 1`

Không có lối tắt, `row` vượt khỏi danh sách các hàng:

In [24]:
def wrong_no_special_case(s: str, numRows: int) -> str:
    rows = [""] * numRows
    row, step = 0, 1
    for c in s:
        rows[row] += c
        if row == 0:
            step = 1
        elif row == numRows - 1:
            step = -1
        row += step
    return "".join(rows)

try:
    wrong_no_special_case("AB", 1)
except IndexError as e:
    print("Lỗi:", type(e).__name__, "-", e)

Lỗi: IndexError - list index out of range


Với công thức chu kỳ, `numRows = 1` còn tệ hơn: `cycle = 0`, và `range(0, n, 0)` báo lỗi `ValueError`.

## Lỗi 2: Dùng `[[]] * numRows`

Như mục 4: mọi hàng dùng chung một list, kết quả sai hoàn toàn.

## Lỗi 3: Đổi hướng sai thời điểm

Nếu cập nhật `row += step` **trước** khi nối ký tự, ký tự đầu tiên rơi vào hàng 1 thay vì hàng 0.

In [25]:
def wrong_move_first(s: str, numRows: int) -> str:
    rows = [""] * numRows
    row, step = 0, 1
    for c in s:
        if row == 0:
            step = 1
        elif row == numRows - 1:
            step = -1
        row += step                     # di chuyển trước...
        rows[row] += c                  # ...rồi mới đặt ký tự: lệch một bước
    return "".join(rows)

print("Sai:", wrong_move_first("PAYPALISHIRING", 3))
print("Đúng:", convert_simulation("PAYPALISHIRING", 3))

Sai: PSIPYAIHRNALIG
Đúng: PAHNAPLSIIGYIR


## Lỗi 4: Quên ký tự thứ hai ở hàng giữa (công thức chu kỳ)

Hàng giữa có **hai** ký tự mỗi chu kỳ (`k + r` và `k + cycle - r`). Hàng đỉnh và đáy chỉ có một — nếu thêm `k + cycle - r` cho hàng đỉnh (`r = 0`), ta sẽ lấy trùng ký tự của chu kỳ kế tiếp.

## Lỗi 5: Đọc theo cột thay vì theo hàng

Đề yêu cầu đọc **từng hàng**. Đọc theo cột sẽ ra lại gần như chuỗi gốc.

# 13. Mẹo nhận diện pattern

Hai ý tưởng có thể dùng lại ở nhiều bài khác:

**1. "Gom theo nhóm" thay vì "vẽ cả bảng"**

Khi đề mô tả một hình/bảng phức tạp nhưng **chỉ hỏi thứ tự đọc theo hàng (hoặc nhóm)**, hãy hỏi: *mỗi phần tử thuộc nhóm nào?* rồi gom vào `groups[key]`.

**2. "Con trỏ nảy" với `step = ±1`**

Khi một chỉ số đi tới biên rồi **quay đầu**, dùng biến hướng:

```python
if pos == 0:
    step = 1
elif pos == last:
    step = -1
pos += step
```

Mẫu này xuất hiện trong các bài mô phỏng chuyển động qua lại, duyệt ma trận theo đường chéo (**#498 Diagonal Traverse**), hay các bài in hình xoắn ốc (**#54 Spiral Matrix**).

# 14. Template tư duy

```python
if numRows == 1 or numRows >= len(s):
    return s

rows = [""] * numRows
row, step = 0, 1

for c in s:
    rows[row] += c
    if row == 0:
        step = 1
    elif row == numRows - 1:
        step = -1
    row += step

return "".join(rows)
```

Ba điều cần nhớ: **xử lý `numRows = 1`**, **đặt ký tự trước rồi mới di chuyển**, **đổi hướng ở đỉnh và đáy**.

# 15. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Không chạy code. Viết `"HELLOWORLD"` theo zigzag với `numRows = 3`:

1. Vẽ hình.
2. Ghi số hàng cho từng ký tự.
3. Kết quả `convert` là gì?

## Bài 2 — Đoán output

```python
print(row_sequence(8, 2))
print(row_sequence(10, 4))
print(2 * 5 - 2)
```

## Bài 3 — Đếm số ký tự mỗi hàng

Viết hàm trả về list độ dài từng hàng (không cần tạo chuỗi).

```python
row_lengths(14, 3)  -> [4, 7, 3]
row_lengths(14, 4)  -> [3, 5, 4, 2]
row_lengths(5, 1)   -> [5]
```

In [26]:
def exercise_3_row_lengths(n: int, numRows: int):
    # TODO:
    # Tạo counts = [0] * numRows
    # Mô phỏng row, step như lời giải chính, mỗi bước counts[row] += 1
    # Nhớ xử lý numRows == 1
    pass


# assert exercise_3_row_lengths(14, 3) == [4, 7, 3]
# assert exercise_3_row_lengths(14, 4) == [3, 5, 4, 2]
# assert exercise_3_row_lengths(5, 1) == [5]

## Bài 4 — Điền chỗ trống: dùng list các list

Viết lại lời giải chính, nhưng mỗi hàng là một **list ký tự** (tránh `+=` trên chuỗi). Cẩn thận cái bẫy `[[]] * n`.

In [27]:
def exercise_4_convert(s: str, numRows: int) -> str:
    # TODO 1: xử lý trường hợp đặc biệt
    # TODO 2: tạo numRows list rỗng ĐỘC LẬP
    # TODO 3: mô phỏng row, step; append ký tự vào đúng hàng
    # TODO 4: ghép tất cả các hàng thành một chuỗi
    pass


# assert exercise_4_convert("PAYPALISHIRING", 3) == "PAHNAPLSIIGYIR"
# assert exercise_4_convert("PAYPALISHIRING", 4) == "PINALSIGYAHRPI"
# assert exercise_4_convert("A", 1) == "A"

## Bài 5 — Biến thể: chỉ số của một hàng

Dùng công thức chu kỳ, viết hàm trả về **danh sách chỉ số** (trong `s`) của các ký tự thuộc hàng `r`.

```python
indices_of_row(14, 4, 0) -> [0, 6, 12]
indices_of_row(14, 4, 1) -> [1, 5, 7, 11, 13]
indices_of_row(14, 4, 3) -> [3, 9]
```

Giả sử `numRows >= 2`.

In [28]:
def exercise_5_indices_of_row(n: int, numRows: int, r: int):
    # TODO
    pass


# assert exercise_5_indices_of_row(14, 4, 0) == [0, 6, 12]
# assert exercise_5_indices_of_row(14, 4, 1) == [1, 5, 7, 11, 13]
# assert exercise_5_indices_of_row(14, 4, 3) == [3, 9]

## Bài 6 — Mini challenge: giải mã zigzag

Bài toán **ngược**: cho chuỗi đã mã hóa `encoded` và `numRows`, hãy khôi phục chuỗi gốc.

```python
decode_zigzag("PAHNAPLSIIGYIR", 3) -> "PAYPALISHIRING"
decode_zigzag("PINALSIGYAHRPI", 4) -> "PAYPALISHIRING"
decode_zigzag("A", 1)              -> "A"
```

In [29]:
def exercise_6_decode_zigzag(encoded: str, numRows: int) -> str:
    # TODO
    pass


# assert exercise_6_decode_zigzag("PAHNAPLSIIGYIR", 3) == "PAYPALISHIRING"
# assert exercise_6_decode_zigzag("PINALSIGYAHRPI", 4) == "PAYPALISHIRING"
# assert exercise_6_decode_zigzag("A", 1) == "A"

# 16. Gợi ý

**Bài 1**
- Mức 1: dãy hàng của 3 hàng là `0, 1, 2, 1, 0, 1, 2, 1, ...`.
- Mức 2: `H`→0, `E`→1, `L`→2, `L`→1, `O`→0, ...
- Mức 3: hàng 0 có 3 ký tự, hàng 1 có 5 ký tự, hàng 2 có 2 ký tự.

**Bài 3**
- Mức 1: giống `convert_simulation` nhưng đếm thay vì nối chuỗi.
- Mức 2: `if numRows == 1: return [n]`.
- Mức 3: `counts[row] += 1` thay cho `rows[row] += c`.

**Bài 4**
- Mức 1: tạo hàng bằng `[[] for _ in range(numRows)]`.
- Mức 2: `rows[row].append(c)`.
- Mức 3: `"".join("".join(r) for r in rows)` hoặc `"".join(c for r in rows for c in r)`.

**Bài 5**
- Mức 1: `cycle = 2 * numRows - 2`, duyệt `k` trong `range(0, n, cycle)`.
- Mức 2: luôn thử `k + r`.
- Mức 3: chỉ hàng giữa (`0 < r < numRows - 1`) mới thử thêm `k + cycle - r`.

**Bài 6**
- Mức 1: nếu biết **mỗi hàng dài bao nhiêu** (Bài 3), ta cắt `encoded` thành từng hàng.
- Mức 2: sau đó mô phỏng lại đường đi zigzag; mỗi bước lấy **ký tự tiếp theo** của hàng `row`.
- Mức 3: giữ `pos = [0] * numRows` là vị trí đã lấy tới trong mỗi hàng.

# 17. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

```text
H   O   L
E L W R D
L   O
```

| Ký tự | H | E | L | L | O | W | O | R | L | D |
|---|---|---|---|---|---|---|---|---|---|---|
| Hàng | 0 | 1 | 2 | 1 | 0 | 1 | 2 | 1 | 0 | 1 |

Kết quả: `"HOL" + "ELWRD" + "LO"` = `"HOLELWRDLO"`.

In [30]:
print_grid(build_grid("HELLOWORLD", 3))
print(sol.convert("HELLOWORLD", 3))

H O L
ELWRD
L O
HOLELWRDLO


## Đáp án Bài 2

In [31]:
print(row_sequence(8, 2))    # [0, 1, 0, 1, 0, 1, 0, 1]
print(row_sequence(10, 4))   # [0, 1, 2, 3, 2, 1, 0, 1, 2, 3]
print(2 * 5 - 2)             # 8: chu kỳ khi numRows = 5

[0, 1, 0, 1, 0, 1, 0, 1]
[0, 1, 2, 3, 2, 1, 0, 1, 2, 3]
8


## Đáp án Bài 3

In [32]:
def answer_3_row_lengths(n: int, numRows: int):
    if numRows == 1:
        return [n]
    counts = [0] * numRows
    row, step = 0, 1
    for _ in range(n):
        counts[row] += 1
        if row == 0:
            step = 1
        elif row == numRows - 1:
            step = -1
        row += step
    return counts

assert answer_3_row_lengths(14, 3) == [4, 7, 3]
assert answer_3_row_lengths(14, 4) == [3, 5, 4, 2]
assert answer_3_row_lengths(5, 1) == [5]
print("Bài 3 PASS")

Bài 3 PASS

## Đáp án Bài 4

In [33]:
def answer_4_convert(s: str, numRows: int) -> str:
    if numRows == 1 or numRows >= len(s):
        return s
    rows = [[] for _ in range(numRows)]
    row, step = 0, 1
    for c in s:
        rows[row].append(c)
        if row == 0:
            step = 1
        elif row == numRows - 1:
            step = -1
        row += step
    return "".join(c for r in rows for c in r)

assert answer_4_convert("PAYPALISHIRING", 3) == "PAHNAPLSIIGYIR"
assert answer_4_convert("PAYPALISHIRING", 4) == "PINALSIGYAHRPI"
assert answer_4_convert("A", 1) == "A"
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [34]:
def answer_5_indices_of_row(n: int, numRows: int, r: int):
    cycle = 2 * numRows - 2
    idx = []
    for k in range(0, n, cycle):
        if k + r < n:
            idx.append(k + r)
        if 0 < r < numRows - 1 and k + cycle - r < n:
            idx.append(k + cycle - r)
    return idx

assert answer_5_indices_of_row(14, 4, 0) == [0, 6, 12]
assert answer_5_indices_of_row(14, 4, 1) == [1, 5, 7, 11, 13]
assert answer_5_indices_of_row(14, 4, 3) == [3, 9]
print("Bài 5 PASS")

Bài 5 PASS


## Đáp án Bài 6

Hai bước:

1. Dùng `answer_3_row_lengths` để biết độ dài mỗi hàng, rồi **cắt** `encoded` thành các hàng.
2. Đi lại đường zigzag; ở mỗi bước, lấy ký tự kế tiếp của hàng `row`.

In [35]:
def answer_6_decode_zigzag(encoded: str, numRows: int) -> str:
    n = len(encoded)
    if numRows == 1 or numRows >= n:
        return encoded

    # Bước 1: cắt encoded thành từng hàng
    lengths = answer_3_row_lengths(n, numRows)
    rows = []
    start = 0
    for length in lengths:
        rows.append(encoded[start:start + length])
        start += length

    # Bước 2: đi lại đường zigzag, lấy lần lượt từ mỗi hàng
    pos = [0] * numRows
    out = []
    row, step = 0, 1
    for _ in range(n):
        out.append(rows[row][pos[row]])
        pos[row] += 1
        if row == 0:
            step = 1
        elif row == numRows - 1:
            step = -1
        row += step
    return "".join(out)

assert answer_6_decode_zigzag("PAHNAPLSIIGYIR", 3) == "PAYPALISHIRING"
assert answer_6_decode_zigzag("PINALSIGYAHRPI", 4) == "PAYPALISHIRING"
assert answer_6_decode_zigzag("A", 1) == "A"

for _ in range(500):
    s = "".join(random.choice("abcXYZ,.") for _ in range(random.randint(1, 25)))
    k = random.randint(1, 10)
    assert answer_6_decode_zigzag(sol.convert(s, k), k) == s
print("Bài 6 PASS (kể cả 500 test mã hóa rồi giải mã)")

Bài 6 PASS (kể cả 500 test mã hóa rồi giải mã)


# 18. Tổng kết

Cốt lõi của bài:

> Hình zigzag chỉ là cách trình bày. Điều thật sự quan trọng là **mỗi ký tự thuộc hàng nào**. Theo dõi `row` nảy giữa `0` và `numRows - 1`, gom ký tự vào đúng hàng, rồi ghép các hàng lại.

---

## Checklist trước khi nộp

- [ ] Có xử lý `numRows == 1`?
- [ ] Tạo các hàng độc lập (không dùng `[[]] * n`)?
- [ ] Đặt ký tự **trước**, di chuyển **sau**?
- [ ] Đổi hướng đúng ở `row == 0` và `row == numRows - 1`?
- [ ] Đã thử `numRows = 1`, `numRows = 2`, `numRows >= len(s)`?

---

## Khi nào nên nghĩ tới kỹ thuật này?

- Đề vẽ một hình/bảng nhưng chỉ hỏi **thứ tự đọc** → gom theo nhóm (`groups[key]`).
- Có một chỉ số chạy tới biên rồi **quay đầu** → biến hướng `step = ±1`.
- Mẫu lặp lại đều đặn → tìm **chu kỳ** và công thức chỉ số.